## Preparation: Python Environment Setup

### Before beginning the activities, participants need to configure their Python environment to ensure that all spatial libraries and the GEE API operate correctly.

#### Instructions:

##### 1. Install Anaconda.
##### 2. Open your Anaconda Prompt and create a new environment.
Enter: conda create -n ai4caf_env python=3.10 -y
##### 3. Activate the environment.
Enter: conda activate 02March2026-training_env
##### 4. Install the Google Earth Engine API, geemap (for UI mapping), and spatial libraries.
Enter: conda install -c conda-forge earthengine-api geemap geopandas rasterio shapely scikit-learn shap -y
##### 5. Install Jupyter Notebook to run the interactive exercises.
Enter: conda install -c conda-forge notebook -y
##### 6. Open Jupyter Notebook
Enter: jupyter notebook
##### 7. Authenticate and initialize Google Earth Engine.

## How to Convert a GEE JS Script to a GEE Python API Script

Transitioning from JavaScript to Python in GEE mostly involves syntax adjustments, as the core Earth Engine functions (ee.Image, ee.FeatureCollection, etc.) are identical. Here are the primary rules for conversion:

1. **Initialization:** Python requires explicit authentication and initialization. You must start your scripts with ee.Authenticate() and ee.Initialize().
2. **Variable Declarations:** Remove the var keyword used in JS. In Python, variables are declared directly (e.g., aoi = ee.Geometry.Polygon(...)).
3. **Booleans and Nulls:** Capitalize boolean and null values. JS true, false, and null become True, False, and None in Python.
4. **Dictionaries:** In JS, dictionary keys don't always need quotes (e.g., {min: 0, max: 100}). In Python, strings must be quoted (e.g., {'min': 0, 'max': 100}).
5. **Functions:** JS inline functions function(img) { return ... } are converted to Python def statements or lambda functions: lambda img: ....
6. **User Interface (UI):** The ui.Panel, ui.Button, and Map.addLayer JS functions do not exist in the standard GEE Python API. To replicate this interactive UI and mapping experience in Python, we use the third-party geemap library.

In [15]:
## Converting the GEE Script (Processing & Mapping)

### This script integrates Sentinel-2, Landsat 8/9, MODIS, and DEM data, calculates indices (NDVI, NDWI, NDBI, EVI), and displays the result on an interactive geemap widget.

import geemap

# 1. Authenticate and Initialize (Run this once per session)
# ee.Authenticate() # Uncomment if running for the very first time
ee.Initialize(project='ee-datandangwork') # Replace with your GCP project ID

# 2. Initialize interactive map (Replaces standard JS Map)
Map = geemap.Map()

# Define a sample AOI (e.g., a province boundary or manual polygon)
aoi = ee.Geometry.Polygon([
  [[124.615, 8.445], [124.610, 8.390], [124.645, 8.390], [124.650, 8.435], [124.615, 8.445]]
])

# 3. Processing Function (Converted from JS getCombinedImages)
def getCombinedImages(aoi_region, maxCloud=20):
    # Sentinel-2 with calculated indices
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi_region)
          .filterDate('2024-01-01', '2024-12-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', maxCloud))
          .median().clip(aoi_region)
          .select(['B2','B3','B4','B8','B11']))
    
    # Landsat 8/9
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
          .filterBounds(aoi_region)
          .filterDate('2024-01-01', '2024-12-31')
          .filter(ee.Filter.lt('CLOUD_COVER', maxCloud))
          .median().clip(aoi_region)
          .multiply(0.0000275).subtract(0.2)
          .select(['SR_B2','SR_B3','SR_B4','SR_B5'])
          .rename(['B2','B3','B4','B8']))
    
    # Combine S2 and L8
    combined = s2.addBands(l8)
    
    # Calculate Indices
    ndvi = combined.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = s2.normalizedDifference(['B3','B8']).rename('NDWI')
    ndbi = s2.normalizedDifference(['B11','B8']).rename('NDBI')
    
    # EVI calculation requires an expression mapping
    evi = s2.expression(
        '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))',
        {'NIR': s2.select('B8'), 'RED': s2.select('B4'), 'BLUE': s2.select('B2')}
    ).rename('EVI')
    
    combined = combined.addBands([ndvi, ndwi, ndbi, evi])
    
    # Add MODIS
    modis = (ee.ImageCollection('MODIS/006/MOD13Q1')
             .filterBounds(aoi_region)
             .filterDate('2024-01-01', '2024-12-31')
             .select(['NDVI','EVI'])
             .map(lambda img: img.multiply(0.0001))
             .median().clip(aoi_region))
    
    combined = combined.addBands(modis)
    
    # Add DEM & Slope
    dem = ee.Image('USGS/SRTMGL1_003').clip(aoi_region)
    slope = ee.Terrain.slope(dem).rename('slope')
    combined = combined.addBands([dem.rename('DEM'), slope])
    
    return combined

# 4. Execute and add to map
processed_image = getCombinedImages(aoi, maxCloud=20)

Map.centerObject(aoi, 11)
Map.addLayer(processed_image.select(['B4', 'B3', 'B2']), {'min': 0, 'max': 0.3}, 'True Color (S2)')
Map.addLayer(processed_image.select('NDVI'), {'min': 0, 'max': 1, 'palette': ['white', 'green']}, 'NDVI Layer')
Map

Map(center=[8.415199403667012, 124.62943650800933], controls=(WidgetControl(options=['position', 'transparent_…

In [55]:
## Other Examples using PSA-Agricultural-Survey.csv
### PSA Survey Data to GEE Features

##### import pandas as pd
import ee
import geemap

# 1. Initialize GEE
# ee.Authenticate()
ee.Initialize(project='ee-datandangwork')
Map = geemap.Map()

# 2. Load the PSA Agricultural Survey Data
# (Ensure the CSV is uploaded to your environment)
df = pd.read_csv('PSA-Agricultural-Survey.csv')

# 3. Data Cleaning: Filter rows that have valid GPS point data
df_points = df.dropna(subset=['crop_location-Latitude', 'crop_location-Longitude']).copy()

# 4. Create a numeric class mapping for crop types (Required for ML Classification)
# The survey includes eggplant, seaweed, calamansi, sweetpotato, tobacco, abaca, tomato, mango
crop_mapping = {
    'eggplant': 0, 'seaweed': 1, 'calamansi': 2, 'sweetpotato': 3, 
    'tobacco': 4, 'abaca': 5, 'tomato': 6, 'mango': 7
}
df_points['class'] = df_points['crop_type'].map(crop_mapping)

# 5. Convert Pandas DataFrame to Earth Engine FeatureCollection
features = []
for index, row in df_points.iterrows():
    # Extract coordinates
    lat = row['crop_location-Latitude']
    lon = row['crop_location-Longitude']
    
    # Create an ee.Geometry.Point
    geom = ee.Geometry.Point([lon, lat])
    
    # Create a feature with properties
    feature = ee.Feature(geom, {
        'crop_type': row['crop_type'],
        'class': row['class'],
        'date_planted': row['date_planted'] if pd.notna(row['date_planted']) else 'Unknown'
    })
    features.append(feature)

survey_fc = ee.FeatureCollection(features)

# 6. Filter for your specific target crop
# Change 'seaweed' to 'mango', 'tomato', 'eggplant', etc., as needed.
target_crop = 'mango' 
filtered_target_fc = survey_fc.filter(ee.Filter.eq('crop_type', target_crop))

# Optional: Add all survey points in a muted color for context
Map.addLayer(survey_fc, {'color': 'gray'}, 'All PSA Survey Points')

# 7. Display the points on the map
Map.centerObject(survey_fc, 8)

# Add the specifically targeted crop in a bright color
Map.addLayer(filtered_target_fc, {'color': 'yellow'}, f'Target Crop: {target_crop.capitalize()}')

Map

Map(center=[8.013039479951676, 122.76446096165442], controls=(WidgetControl(options=['position', 'transparent_…